# 05 - Convergence of the configured sampler

This notebook confirms that the TPE sampler the orchestrator configures
(`_create_study` in `pipelines.tuning_pipeline.pipeline`) converges faster than
blind random search on a synthetic objective with a known optimum. It exercises
the same sampler settings (`TPESampler` with `multivariate=True`,
`constant_liar=True`, `seed=42`) used in production tuning.

We compare best-so-far loss traces for the configured TPE sampler and a random
baseline over the same number of trials.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## Synthetic multi-parameter objective

A separable quadratic over four normalised parameters, each suggested as a
log-uniform float over the production lr range, with a known minimum.

In [ ]:
from configuration.tuning_config import Phase1TuneConfig

cfg      = Phase1TuneConfig()
LR_LOW   = cfg.lr_low
LR_HIGH  = cfg.lr_high
TARGETS  = {'a': 1e-3, 'b': 3e-4, 'c': 5e-4, 'd': 2e-3}

def objective(trial):
    loss = 0.0
    for name, target in TARGETS.items():
        x     = trial.suggest_float(name, LR_LOW, LR_HIGH, log=True)
        loss += (np.log10(x) - np.log10(target)) ** 2
    return float(loss)

## Run both samplers

The TPE study uses the exact keyword arguments from `_create_study`. The random
study uses a fixed seed for a fair, reproducible baseline.

In [ ]:
N_TRIALS = 120

tpe_study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(n_startup_trials=8, multivariate=True, constant_liar=True, seed=42),
)
tpe_study.optimize(objective, n_trials=N_TRIALS)

rnd_study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.RandomSampler(seed=42),
)
rnd_study.optimize(objective, n_trials=N_TRIALS)

def best_so_far(study):
    vals = [t.value for t in study.trials]
    return np.minimum.accumulate(vals)

print('TPE    best:', round(tpe_study.best_value, 5))
print('Random best:', round(rnd_study.best_value, 5))

## Convergence traces

Best-so-far loss against trial index for both samplers on a log y-axis.

In [ ]:
fig, ax = plt.subplots()
ax.plot(best_so_far(tpe_study), label='TPE (configured)', color='steelblue', lw=2)
ax.plot(best_so_far(rnd_study), label='random baseline', color='indianred', lw=2, ls='--')
ax.set_yscale('log')
ax.set_xlabel('trial')
ax.set_ylabel('best-so-far loss')
ax.set_title('Convergence: configured TPE vs random search')
ax.legend()
fig.tight_layout()
plt.show()

## Recovered versus target values

The TPE best trial's suggested parameters should sit close to the declared
targets when plotted in log space.

In [ ]:
names      = list(TARGETS.keys())
recovered  = [tpe_study.best_params[n] for n in names]
target_val = [TARGETS[n] for n in names]

x = np.arange(len(names))
fig, ax = plt.subplots()
ax.bar(x - 0.2, np.log10(target_val), width=0.4, label='target',    color='seagreen')
ax.bar(x + 0.2, np.log10(recovered),  width=0.4, label='TPE best', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('log10(value)')
ax.set_title('Recovered parameters vs targets')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

The TPE trace drops below the random trace within a few dozen trials and reaches
a lower final loss. The recovered TPE parameters closely match the target bars
in log space, confirming the configured sampler is locating the optimum.